In [ ]:
import requests
import os
import time
import sys
import json
from datetime import datetime
from dotenv import load_dotenv

sys.path.append("../Assets")
from Assets.Cities import CITIES_DICT

load_dotenv()

In [ ]:
api_key1 = os.getenv("OPENWEATHER_API_KEY")
api_key2 = os.getenv("WAQIP_API_KEY")

print(f"Weather API Loaded: {api_key1[:5]}..." if api_key1 else "Weather API Missing")
print(f"WAQI API Loaded: {api_key2[:5]}..." if api_key2 else "WAQI API Missing")

In [ ]:
Temp_cities = CITIES_DICT

In [ ]:
path1 = '../Json_data/City.json'
path2 = '../Json_data/Weather_records.json'
path3 = '../Json_data/Aqi_stations.json'
path4 = '../Json_data/Aqi_records.json'
path5 = '../Json_data/Forecast_records.json'

temp1 = '../Json_data/Temp_City.json'
temp2 = '../Json_data/Temp_Weather_records.json'
temp3 = '../Json_data/Temp_Aqi_stations.json'
temp4 = '../Json_data/Temp_Aqi_records.json'
temp5 = '../Json_data/Temp_Forecast_records.json'

In [ ]:
date = datetime.now().strftime("%Y-%m-%d")
forecast_day = datetime.now().day

In [ ]:
def load_existing_data(file_path):
    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        return []

    try:
        with open(file_path, 'r') as f:
            return json.load(f)

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

In [ ]:
def save_data(file_path, data):
    with open(file_path, 'w') as f:
        json.dump(data, f, indent=4)

In [ ]:
def build_existing_keys(data):
    keys = set()

    for record in data:
        city_id = record.get("City_id")
        record_date = record.get("Date")

        if city_id and record_date:
            keys.add((city_id, record_date))

    return keys

In [ ]:
def Weather_error(city_name):
    url = (
        f"http://api.openweathermap.org/data/2.5/weather"
        f"?q={city_name}&appid={api_key1}&units=metric"
    )

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            return response.json()

        elif response.status_code == 404:
            print(f"✗ Weather city not found: {city_name}")

        elif response.status_code == 401:
            print("✗ Invalid weather API key")

        else:
            print(f"✗ Weather API error: {response.status_code}")

        return None

    except requests.exceptions.Timeout:
        print(f"✗ Weather timeout: {city_name}")
        return None

    except requests.exceptions.RequestException as e:
        print(f"✗ Weather network error: {e}")
        return None

In [ ]:
def waqi_error(city_name):
    url = f"https://api.waqi.info/feed/{city_name}/?token={api_key2}"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            return response.json()

        elif response.status_code == 404:
            print(f"✗ AQI city not found: {city_name}")

        elif response.status_code == 401:
            print("✗ Invalid AQI API key")

        else:
            print(f"✗ AQI API error: {response.status_code}")

        return None

    except requests.exceptions.Timeout:
        print(f"✗ AQI timeout: {city_name}")
        return None

    except requests.exceptions.RequestException as e:
        print(f"✗ AQI network error: {e}")
        return None

In [ ]:
print("\nLoading existing files...")

city_data_existing = load_existing_data(path1)
weather_existing = load_existing_data(path2)
aqi_station_existing = load_existing_data(path3)
aqi_record_existing = load_existing_data(path4)
forecast_existing = load_existing_data(path5)

In [ ]:
city_keys = build_existing_keys(city_data_existing)
weather_keys = build_existing_keys(weather_existing)
aqi_station_keys = build_existing_keys(aqi_station_existing)
aqi_record_keys = build_existing_keys(aqi_record_existing)
forecast_keys = build_existing_keys(forecast_existing)

In [ ]:
print("\nFetching API data once per city...\n")

weather_cache = {}
aqi_cache = {}

for city_id, city_name in Temp_cities.items():
    print(f"Fetching: {city_id}: {city_name}")

    weather_cache[city_id] = Weather_error(city_name)
    aqi_cache[city_id] = waqi_error(city_name)

    # time.sleep(0.1)

In [ ]:
new_city_records = []
new_weather_records = []
new_aqi_stations = []
new_aqi_records = []
new_forecasts = []

In [ ]:
for city_id, city_name in Temp_cities.items():

    weather = weather_cache.get(city_id)

    if (city_id, date) not in city_keys and weather:
        city_record = {
            'City_id': city_id,
            'Open_Weather_City_id': weather['id'],
            'City': city_name,
            'Country': weather['sys']['country'],
            'Latitude': weather['coord']['lat'],
            'Longitude': weather['coord']['lon'],
            'Timezone_offset': weather['timezone'],
            'Date': date
        }

        new_city_records.append(city_record)

print(f"New City Records: {len(new_city_records)}")

In [ ]:
for city_id, city_name in Temp_cities.items():

    weather = weather_cache.get(city_id)

    if (city_id, date) not in weather_keys and weather:
        weather_record = {
            "City_id": city_id,
            "Date": date,
            "Temperature": weather['main']['temp'],
            "Feels_like": weather['main']['feels_like'],
            "Temp_min": weather['main']['temp_min'],
            "Temp_max": weather['main']['temp_max'],
            "Pressure": weather['main']['pressure'],
            "Humidity": weather['main']['humidity'],
            "Sea_level": weather['main'].get('sea_level'),
            "Ground_level": weather['main'].get('grnd_level'),
            "Visibility": weather.get('visibility'),
            "Wind_speed": weather['wind'].get('speed'),
            "Wind_degree": weather['wind'].get('deg'),
            "Cloudiness": weather['clouds'].get('all'),
            "Weather_main": weather['weather'][0]['main'],
            "Weather_description": weather['weather'][0]['description'],
            "Weather_icon": weather['weather'][0]['icon'],
            "Sunrise": weather['sys']['sunrise'],
            "Sunset": weather['sys']['sunset']
        }

        new_weather_records.append(weather_record)

print(f"New Weather Records: {len(new_weather_records)}")

In [ ]:
for city_id, city_name in Temp_cities.items():

    waqi = aqi_cache.get(city_id)

    if (
            (city_id, date) not in aqi_station_keys
            and waqi
            and waqi.get("status") == "ok"
    ):
        aqi_station = {
            "Station_id": waqi['data']['idx'],
            "City_id": city_id,
            "Date": date,
            "Station_name": waqi['data']['city']['name'],
            "Latitude": waqi['data']['city']['geo'][0],
            "Longitude": waqi['data']['city']['geo'][1],
            "Station_url": waqi['data']['city']['url'],
            "Is_active": True
        }

        new_aqi_stations.append(aqi_station)

print(f"New AQI Stations: {len(new_aqi_stations)}")

In [ ]:
for city_id, city_name in Temp_cities.items():

    waqi = aqi_cache.get(city_id)

    if (
            (city_id, date) not in aqi_record_keys
            and waqi
            and waqi.get("status") == "ok"
    ):
        iaqi = waqi['data'].get('iaqi', {})

        aqi_record = {
            "City_id": city_id,
            "Date": date,
            "Aqi": waqi['data']['aqi'],
            "Station_id": waqi['data']['idx'],
            "Dominant_pollutant": waqi['data']['dominentpol'],
            "pm25": iaqi.get('pm25', {}).get('v'),
            "pm10": iaqi.get('pm10', {}).get('v'),
            "o3": iaqi.get('o3', {}).get('v'),
            "no2": iaqi.get('no2', {}).get('v'),
            "so2": iaqi.get('so2', {}).get('v'),
            "co": iaqi.get('co', {}).get('v'),
            "Temperature": iaqi.get('t', {}).get('v'),
            "Humidity": iaqi.get('h', {}).get('v'),
            "Pressure": iaqi.get('p', {}).get('v'),
            "Wind_speed": iaqi.get('w', {}).get('v'),
            "Wind_direction": iaqi.get('wd', {}).get('v'),
            "Wind_gust": iaqi.get('wg', {}).get('v'),
            "Dew_point": iaqi.get('dew', {}).get('v')
        }

        new_aqi_records.append(aqi_record)

print(f"New AQI Records: {len(new_aqi_records)}")

In [ ]:
for city_id, city_name in Temp_cities.items():

    waqi = aqi_cache.get(city_id)

    if (
            (city_id, date) not in forecast_keys
            and waqi
            and waqi.get("status") == "ok"
    ):

        try:

            pm25_forecast = waqi['data']['forecast']['daily']['pm25'][0]

            forecast_record = {
                "City_id": city_id,
                "Date": date,
                "Station_id": waqi['data']['idx'],
                "Forecast_day": forecast_day,
                "Metric_type": "pm25",
                "Avg_value": pm25_forecast.get('avg'),
                "Max_value": pm25_forecast.get('max'),
                "Min_value": pm25_forecast.get('min')
            }

            new_forecasts.append(forecast_record)

        except Exception:
            print(f"✗ Forecast missing for {city_name}")

print(f"New Forecast Records: {len(new_forecasts)}")

In [ ]:
city_data_existing.extend(new_city_records)
weather_existing.extend(new_weather_records)
aqi_station_existing.extend(new_aqi_stations)
aqi_record_existing.extend(new_aqi_records)
forecast_existing.extend(new_forecasts)

In [ ]:
save_data(path1, city_data_existing)
save_data(path2, weather_existing)
save_data(path3, aqi_station_existing)
save_data(path4, aqi_record_existing)
save_data(path5, forecast_existing)

save_data(temp1, new_city_records)
save_data(temp2, new_weather_records)
save_data(temp3, new_aqi_stations)
save_data(temp4, new_aqi_records)
save_data(temp5, new_forecasts)

In [ ]:
print("====================================")
print("✓ DATA COLLECTION COMPLETE")
print("====================================")

print(f"Cities Added: {len(new_city_records)}")
print(f"Weather Added: {len(new_weather_records)}")
print(f"AQI Stations Added: {len(new_aqi_stations)}")
print(f"AQI Records Added: {len(new_aqi_records)}")
print(f"Forecast Added: {len(new_forecasts)}")

In [ ]:
def clear_memory():
    variables_to_clear = [
        'weather_cache', 'aqi_cache',
        'new_city_records', 'new_weather_records',
        'new_aqi_stations', 'new_aqi_records', 'new_forecasts',
        'city_data_existing', 'weather_existing',
        'aqi_station_existing', 'aqi_record_existing', 'forecast_existing',
        'city_keys', 'weather_keys', 'aqi_station_keys',
        'aqi_record_keys', 'forecast_keys'
    ]

    for var in variables_to_clear:
        if var in globals():
            del globals()[var]

    import gc
    gc.collect()
    print("Memory cleared!")

clear_memory()